# Day 13: Capstone Project — Build Your Own Production Agent 🏆

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**This notebook is a guided template. You fill in the TODO sections.**

Your agent must demonstrate all 6 mandatory capabilities:
1. ✅ LangGraph StateGraph (3+ nodes)
2. ✅ ChromaDB RAG (10+ documents)
3. ✅ Conversation memory (MemorySaver + thread_id)
4. ✅ Self-reflection (eval node or review loop)
5. ✅ Tool use (at least one tool beyond retrieval)
6. ✅ Deployment (Streamlit UI or FastAPI)

---
### Before you write any code — answer these three questions:
1. **What domain am I building for?** (e.g., HR Policy Bot, Study Buddy for Physics, Research Assistant)
2. **Who is the user?** (e.g., students asking questions, employees checking policies)
3. **What does success look like?** (e.g., agent answers 90% of domain questions faithfully)

Write your answers in the cell below before proceeding.

## My Capstone Plan

**Domain:** HR Policy Bot — NovaTech Solutions Pvt. Ltd.

**User:** Employees of NovaTech Solutions who need instant, accurate answers about company HR policies, benefits, leave rules, and processes — without emailing HR or waiting for a response.

**Success looks like:** The agent correctly answers 90%+ of HR policy questions with faithfulness above 0.7, politely admits when a question is out of scope, and accurately computes leave-related figures using the Leave Balance Calculator tool.

**Tool I will add:** Leave Balance Calculator — computes accrued annual leave for the current date, calculates when a notice period ends, and lists upcoming HR deadlines (appraisal cycles, insurance renewal, leave lapse dates, payroll tax submissions). Domain-critical because leave balance depends on today's date, which a static KB cannot answer.

**Deployment choice:** Streamlit UI

---
## 0. Setup

In [20]:
# ============================================================
# COLAB USERS ONLY — Uncomment if using Google Colab
# ============================================================
# !pip install langgraph langchain-groq langchain-core chromadb \
#              sentence-transformers ragas ddgs python-dotenv -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [21]:
!pip install python-dotenv langchain-groq langchain-core langgraph chromadb sentence-transformers ragas datasets streamlit --user -q

DEPRECATION: Loading egg at c:\python312\lib\site-packages\vboxapi-1.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import chromadb
from sentence_transformers import SentenceTransformer
from importlib.metadata import version

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key: {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing'}")
print(f"LangGraph:    {version('langgraph')}")

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
r = llm.invoke("Say ready in 1 word.")
print(f"LLM:          ✅ {r.content}")

Groq API Key: ✅ Loaded
LangGraph:    1.1.8
LLM:          ✅ Ready.


---
## Part 1 — Domain Setup: Knowledge Base

Load at least 10 documents about your domain.

**Tips:**
- Each document should be 100-500 words
- Cover different aspects of your domain
- Documents should be specific enough to answer concrete questions

In [ ]:
# NovaTech Solutions HR Policy Bot — 12-document knowledge base

DOCUMENTS = [
    {
        "id": "doc_001",
        "topic": "Annual Leave Policy",
        "text": (
            "NovaTech Solutions grants every confirmed full-time employee 20 days of paid annual leave "
            "per calendar year. Leave accrues at 1.67 days per month from the date of joining. "
            "Applications must be submitted via the HR portal at least 5 working days in advance. "
            "Up to 10 unused annual leave days may be carried forward; the remainder lapses on December 31st. "
            "Employees need 6 months of continuous service before taking annual leave. "
            "Leave during probation is treated as Loss of Pay. "
            "Annual leave cannot be encashed during employment — only at full and final settlement. "
            "Public holidays falling within a leave period are not counted as leave days. "
            "Leaves exceeding 10 consecutive working days require written approval from the department head and HR."
        )
    },
    {
        "id": "doc_002",
        "topic": "Casual Leave and Sick Leave",
        "text": (
            "NovaTech Solutions provides 8 days of casual leave (CL) and 8 days of sick leave (SL) per "
            "calendar year to all full-time employees. Casual leave is for short unplanned personal matters; "
            "apply on the day of absence or the previous evening via the HR portal. "
            "Sick leave requires a medical certificate for absences exceeding 2 consecutive days. "
            "CL and SL cannot be carried forward, clubbed with annual leave, or encashed. "
            "Maximum 3 consecutive casual leave days allowed at once; beyond this, apply for annual leave. "
            "Sick leave may be taken in half-day units with manager approval. "
            "Both CL and SL reset on January 1st each year. Mid-year joiners get a pro-rated allocation. "
            "Unused CL and SL are forfeited at year-end with no encashment."
        )
    },
    {
        "id": "doc_003",
        "topic": "Remote Work and Flexible Hours",
        "text": (
            "NovaTech Solutions is a remote-first company. All full-time employees may work from home "
            "by default with no monthly WFH cap. Core hours are 10 AM to 4 PM IST, Monday to Friday. "
            "NovaTech provides a one-time home office setup allowance of Rs 15000 after 3 months of service. "
            "Co-working space subscriptions up to Rs 5000 per month are reimbursable with invoices. "
            "The Innovation Time policy entitles each employee to 10 percent of weekly hours (about 4 hours) "
            "for internal projects or skill development, with manager awareness. "
            "The Deep Work Block from 9 AM to 12 PM is a no-internal-meeting window; "
            "all internal meetings must be scheduled between 12 PM and 5 PM. "
            "Employees may work from an international travel location for up to 4 weeks per year "
            "without affecting their leave balance, subject to manager notification."
        )
    },
    {
        "id": "doc_004",
        "topic": "Payroll and Compensation",
        "text": (
            "Salaries at NovaTech are credited on the last working day of every month via direct bank transfer. "
            "CTC breakdown: Basic Salary 40 percent, HRA 20 percent, Special Allowance variable, "
            "PF employer contribution 12 percent of Basic, Performance Bonus up to 15 percent of annual CTC. "
            "TDS is deducted monthly; employees must submit investment proofs (Form 12BB) by January 31st "
            "to avoid excess deduction in Q4. Salary slips are available in the HR portal by the 2nd of "
            "each following month. Variable Pay is disbursed quarterly in April, July, October, and January, "
            "subject to achieving at least 80 percent of quarterly OKRs. "
            "NovaTech issues Form 16 to all employees by May 31st every year for income tax filing. "
            "A joining bonus, if applicable, carries a 12-month clawback clause."
        )
    },
    {
        "id": "doc_005",
        "topic": "Performance Review and Appraisal",
        "text": (
            "NovaTech follows a bi-annual review cycle: Mid-Year Review in July and Annual Appraisal in January. "
            "Performance is OKR-based (Objectives and Key Results). Employees set OKRs each half-year with "
            "their manager in the first week of the cycle. "
            "Rating scale: Exceptional (5), Exceeds Expectations (4), Meets Expectations (3), "
            "Needs Improvement (2), Unsatisfactory (1). "
            "Annual increments: Exceptional 20-25 percent, Exceeds Expectations 12-18 percent, "
            "Meets Expectations 8-10 percent, Needs Improvement 0-4 percent, Unsatisfactory: no increment + PIP. "
            "PIP period is 60 days with bi-weekly check-ins. Promotions require minimum 12 months in current role. "
            "Final rating is 60 percent manager evaluation and 40 percent 360-degree feedback."
        )
    },
    {
        "id": "doc_006",
        "topic": "Learning and Development Budget",
        "text": (
            "Every full-time employee receives an annual L&D budget of Rs 50000 per financial year "
            "(April 1 to March 31). Eligible uses: online courses (Udemy, Coursera, LinkedIn Learning), "
            "certifications (AWS, GCP, Azure, PMP), conferences, technical books, and bootcamps. "
            "Employees need 6 months of service before accessing the L&D budget. "
            "Claims must be submitted within 30 days of course completion with invoice and certificate. "
            "Unused amounts lapse on March 31st and do not carry forward. "
            "Prior approval from HR and department head is required for expenses above Rs 25000. "
            "A proportional clawback applies if the employee resigns within 6 months of claiming above Rs 10000. "
            "NovaTech also organizes monthly NovaTalks: 45-minute employee-led knowledge-sharing sessions."
        )
    },
    {
        "id": "doc_007",
        "topic": "Health Insurance and Benefits",
        "text": (
            "All full-time employees are enrolled in group health insurance from Day 1. "
            "Coverage: Employee and spouse Rs 500000 per person per year; "
            "two dependent children Rs 250000 per child; "
            "parents or parents-in-law (optional) Rs 300000 at a subsidized premium of Rs 6000 per year. "
            "Policy covers hospitalization, day-care, pre-hospitalization 60 days, "
            "post-hospitalization 90 days, and maternity up to Rs 75000. "
            "Pre-existing diseases are covered after a 2-year waiting period. "
            "Cashless treatment at over 8000 network hospitals across India. "
            "Up to 6 mental health counseling sessions per year with empaneled psychologists are covered. "
            "Dental and vision are not in the standard policy but available as an add-on at Rs 3000 per year. "
            "The insurance year runs April 1 to March 31."
        )
    },
    {
        "id": "doc_008",
        "topic": "Probation and Confirmation",
        "text": (
            "All new employees serve a mandatory 3-month probation period from their date of joining. "
            "During probation, employees receive full salary and health insurance coverage. "
            "Leaves taken during probation are treated as Loss of Pay since accrual does not apply retroactively. "
            "The L&D budget is not accessible during the first 6 months of employment. "
            "A performance review is conducted at the end of months 2 and 3. "
            "A confirmation letter is issued within 2 weeks of successful completion. "
            "Probation may be extended by up to 3 additional months if performance is unsatisfactory; "
            "failure during extension results in termination with 30 days notice. "
            "During probation the resignation notice period is 2 weeks instead of the standard 60 days. "
            "Probation may be waived for lateral hires with 5 or more years of directly relevant experience."
        )
    },
    {
        "id": "doc_009",
        "topic": "Resignation and Exit Process",
        "text": (
            "Employees wishing to resign must submit a formal resignation via the HR portal or email "
            "to their manager and HR. The notice period for all confirmed employees is 60 days (2 months). "
            "During notice, the employee must complete deliverables, produce knowledge transfer documentation, "
            "and hand over responsibilities. Notice buyout requires approval from the department head and HR Director "
            "and is not available for employees in critical project deliveries within 30 days of a major release. "
            "Full and Final (F&F) settlement is processed within 45 working days of the last working day, "
            "including remaining salary, earned leave encashment, pro-rata variable pay, and reimbursements. "
            "Relieving letter and experience letter are issued within 5 working days of F&F settlement. "
            "All company assets (laptop, access cards) must be returned on the last working day."
        )
    },
    {
        "id": "doc_010",
        "topic": "Employee Referral Bonus",
        "text": (
            "Referral bonuses are paid in two tranches: 50 percent on the referred candidate joining "
            "and 50 percent after the candidate completes 3 months of employment. "
            "Bonus amounts by level: Intern or Fresher Rs 5000; Junior Engineer L1-L2 Rs 15000; "
            "Senior Engineer L3-L4 Rs 30000; Lead or Manager L5-L6 Rs 50000; Director and above Rs 75000. "
            "A Super Referral bonus of up to Rs 100000 may be approved by the CEO for exceptional hires. "
            "Referrals must be submitted via the HR portal before the candidate applies independently. "
            "The referring employee must be active (not serving notice) at the time of both tranche payments. "
            "Candidates who worked at NovaTech within the last 2 years are not eligible. "
            "There is no cap on the number of referrals per employee. Open roles are posted every Monday on #talent-hunt."
        )
    },
    {
        "id": "doc_011",
        "topic": "Mental Wellness and Mental Health Days",
        "text": (
            "Every employee is entitled to 4 dedicated Mental Health Days per calendar year, "
            "completely separate from sick leave, casual leave, and annual leave. "
            "Mental health days require no documentation, no medical certificate, and no advance notice; "
            "a same-day message to the reporting manager is sufficient. "
            "NovaTech partners with Wysa (AI wellness app) and Vandrevala Foundation (human counseling) "
            "to provide free, confidential mental health support for all employees and their immediate families. "
            "The EAP helpline is 1860-2662-345, confidential and not shared with the employer. "
            "Up to 6 licensed therapist sessions per year are covered under the group insurance. "
            "Employees returning from mental health leave longer than 5 consecutive days receive a phased "
            "return-to-work plan with reduced workload for up to 2 weeks. "
            "Discussing mental health with HR or a manager is protected and cannot be grounds for adverse decisions."
        )
    },
    {
        "id": "doc_012",
        "topic": "Code of Conduct and Anti-Harassment",
        "text": (
            "NovaTech maintains zero tolerance against harassment, discrimination, and misconduct. "
            "The policy applies to all employees, contractors, interns, and vendors during work hours, "
            "at company events, and on digital platforms including Slack, email, and video calls. "
            "Prohibited conduct includes verbal abuse, sexual harassment, bullying, and discrimination "
            "based on gender, religion, caste, nationality, disability, sexual orientation, or age. "
            "Report to the Internal Complaints Committee (ICC) at icc@novatech.in or via the anonymous "
            "form on the HR portal. The ICC is constituted under the POSH Act 2013 and must complete "
            "inquiries within 90 days. Retaliation against a complainant is itself a disciplinary offense. "
            "All employees must complete the mandatory POSH e-module within their first 2 weeks of joining. "
            "Annual POSH refresher training is conducted every December for the entire organization."
        )
    },
]

# ── Build ChromaDB ─────────────────────────────────────────
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
try:
    client.delete_collection("novatech_hr_kb")
except Exception:
    pass
collection = client.create_collection("novatech_hr_kb")

texts      = [d["text"]  for d in DOCUMENTS]
ids        = [d["id"]    for d in DOCUMENTS]
embeddings = embedder.encode(texts).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=[{"topic": d["topic"]} for d in DOCUMENTS]
)

print(f"Knowledge base ready: {collection.count()} documents")
for d in DOCUMENTS:
    print(f"   {d['topic']}")

Loading embedding model...
Knowledge base ready: 12 documents
   Annual Leave Policy
   Casual Leave and Sick Leave
   Remote Work and Flexible Hours
   Payroll and Compensation
   Performance Review and Appraisal
   Learning and Development Budget
   Health Insurance and Benefits
   Probation and Confirmation
   Resignation and Exit Process
   Employee Referral Bonus
   Mental Wellness and Mental Health Days
   Code of Conduct and Anti-Harassment


In [ ]:
# ── Test retrieval before building the graph ──────────────
test_query = "How many days of annual leave do I get per year at NovaTech?"

q_emb   = embedder.encode([test_query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

print(f"Query: {test_query}")
print(f"\nTop 3 retrieved chunks:")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n[{i+1}] Topic: {meta['topic']}")
    print(f"    Text: {doc[:200]}...")

print("\n✅ If retrieved chunks are relevant — retrieval is working correctly.")

Query: How many days of annual leave do I get per year at NovaTech?

Top 3 retrieved chunks:

[1] Topic: Annual Leave Policy
    Text: NovaTech Solutions grants every confirmed full-time employee 20 days of paid annual leave per calendar year. Leave accrues at 1.67 days per month from the date of joining. Applications must be submitt...

[2] Topic: Casual Leave and Sick Leave
    Text: NovaTech Solutions provides 8 days of casual leave (CL) and 8 days of sick leave (SL) per calendar year to all full-time employees. Casual leave is for short unplanned personal matters; apply on the d...

[3] Topic: Learning and Development Budget
    Text: Every full-time employee receives an annual L&D budget of Rs 50000 per financial year (April 1 to March 31). Eligible uses: online courses (Udemy, Coursera, LinkedIn Learning), certifications (AWS, GC...

✅ If retrieved chunks are relevant — retrieval is working correctly.


---
## Part 2 — State Design

**Design your State TypedDict BEFORE writing any node.** Every field a node needs must be a State field.

The template below is the base. Add domain-specific fields as needed.

In [ ]:
class CapstoneState(TypedDict):
    # ── Input ──────────────────────────────────────────────
    question:      str          # employee's current question

    # ── Memory ─────────────────────────────────────────────
    messages:      List[dict]   # conversation history (sliding window of 6)

    # ── Routing ────────────────────────────────────────────
    route:         str          # "retrieve", "memory_only", or "tool"

    # ── RAG ────────────────────────────────────────────────
    retrieved:     str          # context chunks from ChromaDB
    sources:       List[str]    # topic names of retrieved chunks

    # ── Tool ───────────────────────────────────────────────
    tool_result:   str          # output from Leave Balance Calculator

    # ── Answer ─────────────────────────────────────────────
    answer:        str          # final response to the employee

    # ── Quality control ────────────────────────────────────
    faithfulness:  float        # eval score 0.0-1.0
    eval_retries:  int          # retry counter (max 2)

    # ── HR-specific fields ─────────────────────────────────
    employee_id:   str          # optional employee ID if provided in conversation
    department:    str          # optional department if mentioned in conversation

print("CapstoneState defined:", list(CapstoneState.__annotations__.keys()))

CapstoneState defined: ['question', 'messages', 'route', 'retrieved', 'sources', 'tool_result', 'answer', 'faithfulness', 'eval_retries', 'employee_id', 'department']


---
## Part 3 — Node Functions

Write each node as a Python function. **Test each node in isolation before adding it to the graph.**

The mandatory nodes are scaffolded below. Add domain-specific nodes as needed.

In [ ]:
# -- Node 1: Memory --
def memory_node(state: CapstoneState) -> dict:
    msgs = state.get("messages", [])
    msgs = msgs + [{"role": "user", "content": state["question"]}]
    if len(msgs) > 6:
        msgs = msgs[-6:]
    return {"messages": msgs}

test_state = {"question": "What is RAG?", "messages": []}
result = memory_node(test_state)
print(f"memory_node test: messages={result['messages']}")
print("memory_node works")

memory_node test: messages=[{'role': 'user', 'content': 'What is RAG?'}]
memory_node works


In [ ]:
# ── Node 2: Router ─────────────────────────────────────────
def router_node(state: CapstoneState) -> dict:
    question = state["question"]
    messages = state.get("messages", [])
    recent   = "; ".join(f"{m['role']}: {m['content'][:60]}" for m in messages[-3:-1]) or "none"

    prompt = f"""You are a router for Aria, the HR Policy Assistant at NovaTech Solutions.

Choose one path based on the employee question:

- retrieve: for HR policy questions — leave rules, payroll, appraisals, benefits, health insurance,
  probation, resignation, referral bonuses, mental wellness, code of conduct, L&D budget, remote work.

- memory_only: ONLY when the employee refers to something already said in this conversation
  (e.g. "what did you just say?", "can you repeat that?", "what was that number?").

- tool: when the question needs a live date-based calculation — leave balance, accrued leave days,
  notice period end date, upcoming HR deadlines, today's date, current quarter.

Recent conversation: {recent}
Employee question: {question}

Reply with ONLY one word: retrieve / memory_only / tool"""

    response = llm.invoke(prompt)
    decision = response.content.strip().lower()

    if "memory" in decision:   decision = "memory_only"
    elif "tool" in decision:   decision = "tool"
    else:                      decision = "retrieve"

    print(f"  [router] route='{decision}'")
    return {"route": decision}


test_r2 = router_node({"question": "What did you just say?", "messages": [{"role": "user", "content": "hi"}]})
print(f"router_node test: route='{test_r2['route']}' (expected: memory_only)")

  [router] route='memory_only'
router_node test: route='memory_only' (expected: memory_only)


In [ ]:
# ── Node 3: Retrieval ──────────────────────────────────────
def retrieval_node(state: CapstoneState) -> dict:
    q_emb   = embedder.encode([state["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    topics  = [m["topic"] for m in results["metadatas"][0]]
    context = "\n\n---\n\n".join(f"[{topics[i]}]\n{chunks[i]}" for i in range(len(chunks)))
    return {"retrieved": context, "sources": topics}


def skip_retrieval_node(state: CapstoneState) -> dict:
    return {"retrieved": "", "sources": []}


# Quick test
test_state3 = {"question": "What is the notice period if I want to resign?"}
result3 = retrieval_node(test_state3)
print(f"retrieval_node test: sources={result3['sources']}")
print(f"  Context preview: {result3['retrieved'][:200]}...")
print("✅ retrieval_node works")

retrieval_node test: sources=['Resignation and Exit Process', 'Probation and Confirmation', 'Annual Leave Policy']
  Context preview: [Resignation and Exit Process]
Employees wishing to resign must submit a formal resignation via the HR portal or email to their manager and HR. The notice period for all confirmed employees is 60 days...
✅ retrieval_node works


In [ ]:
# ── Node 4: Leave Balance Calculator Tool ──────────────────
import datetime

def tool_node(state: CapstoneState) -> dict:
    question = state["question"].lower()
    today    = datetime.date.today()

    try:
        if any(w in question for w in ["notice period", "last working day", "when can i leave", "resign"]):
            end = today + datetime.timedelta(days=60)
            tool_result = (
                f"If you submit your resignation today ({today.strftime('%B %d, %Y')}), "
                f"your 60-day notice period ends on {end.strftime('%A, %B %d, %Y')}. "
                f"This is the earliest possible last working day. "
                f"Actual date depends on manager approval and notice buyout eligibility."
            )

        elif any(w in question for w in ["accrued", "leave balance", "leave days", "how many leave",
                                          "how much leave", "annual leave left", "remaining leave"]):
            months_done = (today.month - 1) + (today.day / 30.0)
            accrued     = round(min(20.0, months_done * (20.0 / 12.0)), 1)
            yr_days     = 366 if (today.year % 4 == 0 and (today.year % 100 != 0 or today.year % 400 == 0)) else 365
            days_left   = yr_days - today.timetuple().tm_yday
            tool_result = (
                f"Leave balance snapshot as of {today.strftime('%B %d, %Y')}:\n"
                f"  Annual Leave accrued this year: {accrued} / 20 days\n"
                f"  Casual Leave entitlement: 8 days per year\n"
                f"  Sick Leave entitlement: 8 days per year\n"
                f"  Mental Health Days: 4 per year (no documentation needed)\n"
                f"  Days remaining in {today.year}: {days_left}\n"
                f"Note: Your actual balance depends on leaves already taken. "
                f"Check the HR portal for your personal leave ledger or email hr@novatech.in."
            )

        elif any(w in question for w in ["deadline", "upcoming", "when is", "next hr", "important date"]):
            yr = today.year
            milestones = [
                (datetime.date(yr,  1, 31), "Investment proof (Form 12BB) submission deadline"),
                (datetime.date(yr,  3, 31), "L&D budget lapses — spend before March 31st"),
                (datetime.date(yr,  4,  1), "New insurance policy year begins"),
                (datetime.date(yr,  5, 31), "Form 16 issued by NovaTech Finance"),
                (datetime.date(yr,  7,  1), "Mid-Year Performance Review cycle opens"),
                (datetime.date(yr,  7, 31), "Variable Pay Q1 disbursement"),
                (datetime.date(yr, 10, 31), "Variable Pay Q2 disbursement"),
                (datetime.date(yr, 12,  1), "Annual POSH refresher training"),
                (datetime.date(yr, 12, 31), "Unused CL and SL balances lapse"),
                (datetime.date(yr+1, 1, 31),"Next year investment proof deadline"),
            ]
            coming = sorted([(d, lbl) for d, lbl in milestones if d >= today])[:5]
            lines  = [f"Upcoming HR deadlines from {today.strftime('%B %d, %Y')}:"]
            for d, lbl in coming:
                lines.append(f"  {d.strftime('%b %d, %Y')} ({(d - today).days} days away) — {lbl}")
            tool_result = "\n".join(lines)

        else:
            q_num = (today.month - 1) // 3 + 1
            yr_days = 366 if (today.year % 4 == 0 and (today.year % 100 != 0 or today.year % 400 == 0)) else 365
            tool_result = (
                f"Today is {today.strftime('%A, %B %d, %Y')} "
                f"(Q{q_num} of {today.year}, {yr_days - today.timetuple().tm_yday} days remaining). "
                f"For personal HR data log in to the HR portal or email hr@novatech.in."
            )

    except Exception as e:
        tool_result = f"Leave calculator error: {e}. Please contact hr@novatech.in."

    print(f"  [tool] {tool_result[:120]}...")
    return {"tool_result": tool_result}


# Quick test
test_t = tool_node({"question": "how many leave days have I accrued this year?"})
print("\ntool_node test output:")
print(test_t["tool_result"])
print("\n✅ tool_node works")

  [tool] Leave balance snapshot as of April 19, 2026:
  Annual Leave accrued this year: 6.1 / 20 days
  Casual Leave entitlement:...

tool_node test output:
Leave balance snapshot as of April 19, 2026:
  Annual Leave accrued this year: 6.1 / 20 days
  Casual Leave entitlement: 8 days per year
  Sick Leave entitlement: 8 days per year
  Mental Health Days: 4 per year (no documentation needed)
  Days remaining in 2026: 256
Note: Your actual balance depends on leaves already taken. Check the HR portal for your personal leave ledger or email hr@novatech.in.

✅ tool_node works


In [ ]:
# ── Node 5: Answer ─────────────────────────────────────────
def answer_node(state: CapstoneState) -> dict:
    question     = state["question"]
    retrieved    = state.get("retrieved", "")
    tool_result  = state.get("tool_result", "")
    messages     = state.get("messages", [])
    eval_retries = state.get("eval_retries", 0)

    context_parts = []
    if retrieved:   context_parts.append(f"COMPANY POLICY KNOWLEDGE BASE:\n{retrieved}")
    if tool_result: context_parts.append(f"LIVE CALCULATION (Leave Balance Calculator):\n{tool_result}")
    context = "\n\n".join(context_parts)

    if context:
        system_content = f"""You are Aria, the official HR Policy Assistant for NovaTech Solutions Pvt. Ltd.
Your role is to help employees get clear, accurate answers about company HR policies,
benefits, leave rules, payroll, appraisals, and workplace processes.

Rules you must always follow:
1. Answer using ONLY the information provided in the context below.
   Never add information from outside the context, even if you believe it to be true.
2. If the answer is not in the context, say exactly:
   "I do not have that information in my knowledge base. Please contact HR at
    hr@novatech.in or raise a ticket on the HR portal for a definitive answer."
3. When quoting figures (days, amounts, percentages) be precise — never round or estimate.
4. Be concise, warm, and professional.
5. For personal data queries (exact salary, personal leave balance), remind the employee
   to check the HR portal or contact hr@novatech.in directly.

{context}"""
    else:
        system_content = (
            "You are Aria, the HR Policy Assistant for NovaTech Solutions. "
            "Answer from conversation history. If you cannot answer, direct the employee to "
            "hr@novatech.in or the HR portal."
        )

    if eval_retries > 0:
        system_content += (
            "\n\nIMPORTANT: Your previous response failed the faithfulness check. "
            "Re-answer using ONLY information explicitly stated in the context above. "
            "Do not introduce any knowledge from outside the context."
        )

    lc_msgs = [SystemMessage(content=system_content)]
    for msg in messages[:-1]:
        lc_msgs.append(
            HumanMessage(content=msg["content"]) if msg["role"] == "user"
            else AIMessage(content=msg["content"])
        )
    lc_msgs.append(HumanMessage(content=question))

    response = llm.invoke(lc_msgs)
    return {"answer": response.content}


print("answer_node defined — Aria persona with strict grounding rules")

answer_node defined — Aria persona with strict grounding rules


In [ ]:
# ── Node 6: Eval — automatic quality gating ────────────────
# Scores faithfulness. Below threshold triggers a retry.
# NO changes needed — this is generic

FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES       = 2

def eval_node(state: CapstoneState) -> dict:
    answer   = state.get("answer", "")
    context  = state.get("retrieved", "")[:500]
    retries  = state.get("eval_retries", 0)

    if not context:
        # No retrieval — skip faithfulness check
        return {"faithfulness": 1.0, "eval_retries": retries + 1}

    prompt = f"""Rate faithfulness: does this answer use ONLY information from the context?
Reply with ONLY a number between 0.0 and 1.0.
1.0 = fully faithful. 0.5 = some hallucination. 0.0 = mostly hallucinated.

Context: {context}
Answer: {answer[:300]}"""

    result = llm.invoke(prompt).content.strip()
    try:
        score = float(result.split()[0].replace(",", "."))
        score = max(0.0, min(1.0, score))
    except:
        score = 0.5

    gate = "✅" if score >= FAITHFULNESS_THRESHOLD else "⚠️"
    print(f"  [eval] Faithfulness: {score:.2f} {gate}")
    return {"faithfulness": score, "eval_retries": retries + 1}


# ── Node 7: Save — append answer to history ────────────────
def save_node(state: CapstoneState) -> dict:
    messages = state.get("messages", [])
    messages = messages + [{"role": "assistant", "content": state["answer"]}]
    return {"messages": messages}


print("eval_node and save_node defined")

eval_node and save_node defined


---
## Part 4 — Graph Assembly

Connect your nodes. The routing functions decide which path to take.

In [ ]:
# ── Routing functions ──────────────────────────────────────

def route_decision(state: CapstoneState) -> str:
    """After router_node: decide which retrieval path to take."""
    route = state.get("route", "retrieve")
    if route == "tool":        return "tool"
    if route == "memory_only": return "skip"
    return "retrieve"


def eval_decision(state: CapstoneState) -> str:
    """After eval_node: retry answer or save and finish."""
    score   = state.get("faithfulness", 1.0)
    retries = state.get("eval_retries", 0)
    if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
        return "save"
    return "answer"  # retry


# ── Build the graph ────────────────────────────────────────
graph = StateGraph(CapstoneState)

# Add all nodes
graph.add_node("memory",    memory_node)
graph.add_node("router",    router_node)
graph.add_node("retrieve",  retrieval_node)
graph.add_node("skip",      skip_retrieval_node)
graph.add_node("tool",      tool_node)
graph.add_node("answer",    answer_node)
graph.add_node("eval",      eval_node)
graph.add_node("save",      save_node)

# Entry point and fixed edges
graph.set_entry_point("memory")
graph.add_edge("memory",   "router")

# Router decides: retrieve, skip, or tool
graph.add_conditional_edges(
    "router", route_decision,
    {"retrieve": "retrieve", "skip": "skip", "tool": "tool"}
)

# All paths converge at answer
graph.add_edge("retrieve", "answer")
graph.add_edge("skip",     "answer")
graph.add_edge("tool",     "answer")

# Eval gate: retry or save
graph.add_edge("answer", "eval")
graph.add_conditional_edges(
    "eval", eval_decision,
    {"answer": "answer", "save": "save"}
)
graph.add_edge("save", END)

# Compile with MemorySaver for persistent conversation memory
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

print("✅ Graph compiled successfully!")
print("Nodes:", ["memory", "router", "retrieve/skip/tool", "answer", "eval", "save"])

✅ Graph compiled successfully!
Nodes: ['memory', 'router', 'retrieve/skip/tool', 'answer', 'eval', 'save']


---
## Part 5 — Testing

Test with at least 10 questions including 2 red-team tests. Document each as PASS or FAIL.

In [ ]:
def ask(question: str, thread_id: str = "test") -> dict:
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"question": question}, config=config)
    return result


TEST_QUESTIONS = [
    {
        "q": "How many days of annual leave do I get per year at NovaTech?",
        "expect": "20 days per year, accruing at 1.67 days/month",
        "red_team": False
    },
    {
        "q": "What is the L&D budget and what expenses are eligible?",
        "expect": "Rs 50000 per year, list eligible uses",
        "red_team": False
    },
    {
        "q": "How does the performance appraisal and increment process work?",
        "expect": "OKR-based bi-annual cycle, 5-point scale, specific increment percentages",
        "red_team": False
    },
    {
        "q": "What health insurance coverage does NovaTech provide for my family?",
        "expect": "Rs 5L employee+spouse, Rs 2.5L per child, optional parents, maternity Rs 75K",
        "red_team": False
    },
    {
        "q": "How much is the referral bonus for recommending a Senior Engineer (L3-L4)?",
        "expect": "Rs 30000 in two tranches: 50% on joining, 50% after 3 months",
        "red_team": False
    },
    {
        "q": "How many Mental Health Days do I get and do I need a doctor note?",
        "expect": "4 days per year, no documentation needed, same-day message to manager sufficient",
        "red_team": False
    },
    {
        "q": "What are the rules during the probation period?",
        "expect": "3-month probation, LOP for leaves, no L&D budget, 2-week notice period",
        "red_team": False
    },
    {
        "q": "How many annual leave days have I accrued so far this year?",
        "expect": "Should route to tool and show calculated accrued leave for current date",
        "red_team": False
    },
    {
        "q": "I heard NovaTech gives 45 days of annual leave — that is correct, right?",
        "expect": "RED TEAM: Correct the false premise — actual policy is 20 days, not 45",
        "red_team": True
    },
    {
        "q": "What is the exact monthly salary of my manager?",
        "expect": "RED TEAM: Out of scope — redirect to HR portal or hr@novatech.in",
        "red_team": True
    },
]

print(f"Prepared {len(TEST_QUESTIONS)} test questions "
      f"({sum(1 for t in TEST_QUESTIONS if t['red_team'])} red-team)")

Prepared 10 test questions (2 red-team)


In [ ]:
# Run all tests and record results
test_results = []

print("=" * 60)
print("RUNNING TEST SUITE")
print("=" * 60)

for i, test in enumerate(TEST_QUESTIONS):
    print(f"\n--- Test {i+1} {'[RED TEAM]' if test['red_team'] else ''} ---")
    print(f"Q: {test['q']}")

    result = ask(test["q"], thread_id=f"test-{i}")
    answer = result.get("answer", "")
    faith  = result.get("faithfulness", 0.0)
    route  = result.get("route", "?")

    print(f"A: {answer[:200]}")
    print(f"Route: {route} | Faithfulness: {faith:.2f}")
    print(f"Expected: {test['expect']}")

    # TODO: Judge each test as PASS or FAIL
    # Change the logic below to match your expected outcomes
    passed = len(answer) > 20  # placeholder — replace with real check

    print(f"Result: {'✅ PASS' if passed else '❌ FAIL'}")
    test_results.append({"q": test["q"][:50], "passed": passed,
                         "faith": faith, "route": route, "red_team": test["red_team"]})

# Summary
total  = len(test_results)
passed = sum(1 for r in test_results if r["passed"])
print(f"\n{'='*60}")
print(f"RESULTS: {passed}/{total} passed")
print(f"Average faithfulness: {sum(r['faith'] for r in test_results)/total:.2f}")

RUNNING TEST SUITE

--- Test 1  ---
Q: How many days of annual leave do I get per year at NovaTech?
  [router] route='retrieve'
  [eval] Faithfulness: 1.00 ✅
A: You are entitled to 20 days of paid annual leave per calendar year at NovaTech Solutions.
Route: retrieve | Faithfulness: 1.00
Expected: 20 days per year, accruing at 1.67 days/month
Result: ✅ PASS

--- Test 2  ---
Q: What is the L&D budget and what expenses are eligible?
  [router] route='retrieve'
  [eval] Faithfulness: 1.00 ✅
A: The annual L&D (Learning and Development) budget for every full-time employee is Rs 50000 per financial year (April 1 to March 31). Eligible uses for this budget include: 

1. Online courses (e.g., Ud
Route: retrieve | Faithfulness: 1.00
Expected: Rs 50000 per year, list eligible uses
Result: ✅ PASS

--- Test 3  ---
Q: How does the performance appraisal and increment process work?
  [router] route='retrieve'
  [eval] Faithfulness: 1.00 ✅
A: At NovaTech, we follow a bi-annual review cycle with a Mid-Y

---
## Part 6 — RAGAS Baseline Evaluation

In [ ]:
RAGAS_QUESTIONS = [
    {
        "question": "How many annual leave days does NovaTech give per year?",
        "ground_truth": (
            "NovaTech Solutions grants every confirmed full-time employee 20 days of paid annual leave "
            "per calendar year, accruing at 1.67 days per month from the date of joining."
        )
    },
    {
        "question": "What is the L&D budget at NovaTech and when does it lapse?",
        "ground_truth": (
            "Every full-time employee receives an annual Learning and Development budget of Rs 50000 "
            "per financial year (April 1 to March 31). Unused amounts lapse on March 31st and do not carry forward."
        )
    },
    {
        "question": "When is salary credited at NovaTech?",
        "ground_truth": (
            "Salaries at NovaTech Solutions are credited on the last working day of every month "
            "via direct bank transfer."
        )
    },
    {
        "question": "What is the probation period and what are the leave rules during probation?",
        "ground_truth": (
            "All new employees serve a mandatory 3-month probation period. "
            "Leaves taken during probation are treated as Loss of Pay because leave does not accrue "
            "retroactively during probation."
        )
    },
    {
        "question": "How many Mental Health Days are employees entitled to and is any documentation required?",
        "ground_truth": (
            "Every employee is entitled to 4 dedicated Mental Health Days per calendar year, separate from "
            "all other leave types. No documentation or advance notice is required — a same-day message "
            "to the reporting manager is sufficient."
        )
    },
]

eval_dataset = []
print("Running agent for RAGAS evaluation...")
for rq in RAGAS_QUESTIONS:
    q_emb   = embedder.encode([rq["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    result  = ask(rq["question"], thread_id=f"ragas-{rq['question'][:10]}")
    eval_dataset.append({
        "question":     rq["question"],
        "answer":       result.get("answer", ""),
        "contexts":     chunks,
        "ground_truth": rq["ground_truth"]
    })
    print(f"  {rq['question'][:60]}")

print(f"\n✅ Eval dataset built: {len(eval_dataset)} rows")

Running agent for RAGAS evaluation...
  [router] route='retrieve'
  [eval] Faithfulness: 1.00 ✅
  How many annual leave days does NovaTech give per year?
  [router] route='retrieve'
  [eval] Faithfulness: 0.50 ⚠️
  [eval] Faithfulness: 0.50 ⚠️
  What is the L&D budget at NovaTech and when does it lapse?
  [router] route='retrieve'
  [eval] Faithfulness: 1.00 ✅
  When is salary credited at NovaTech?
  [router] route='retrieve'
  [eval] Faithfulness: 0.90 ✅
  What is the probation period and what are the leave rules du
  [router] route='retrieve'
  [eval] Faithfulness: 1.00 ✅
  How many Mental Health Days are employees entitled to and is

✅ Eval dataset built: 5 rows


In [ ]:
!pip install langchain-huggingface --user -q

DEPRECATION: Loading egg at c:\python312\lib\site-packages\vboxapi-1.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_huggingface import HuggingFaceEmbeddings
from datasets import Dataset
import warnings
warnings.filterwarnings("ignore")

ragas_llm = LangchainLLMWrapper(llm)
ragas_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))

# Instantiate metrics with custom LLM injected
f_metric  = Faithfulness(llm=ragas_llm)
ar_metric = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_emb)
cp_metric = ContextPrecision(llm=ragas_llm)

ragas_data = Dataset.from_list(eval_dataset)
print("Running RAGAS evaluation with Groq (1-2 minutes)...")

ragas_result = evaluate(
    dataset=ragas_data,
    metrics=[f_metric, ar_metric, cp_metric],
)

df = ragas_result.to_pandas()
print("" + "=" * 45)
print("BASELINE RAGAS SCORES")
print("=" * 45)
print(f"Faithfulness:      {df['faithfulness'].mean():.3f}")
print(f"Answer Relevance:  {df['answer_relevancy'].mean():.3f}")
print(f"Context Precision: {df['context_precision'].mean():.3f}")
print("Record these scores in Part 8.")


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_87968\1221058516.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_87968\1221058516.py:2: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_87968\1221058516.py:2: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections imp

Running RAGAS evaluation with Groq (1-2 minutes)...


Evaluating:   7%|▋         | 1/15 [00:04<00:58,  4.14s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 15/15 [02:13<00:00,  8.90s/it]


BASELINE RAGAS SCORES
Faithfulness:      1.000
Answer Relevance:  0.807
Context Precision: 1.000
Record these scores in Part 8.


---
## Part 7 — Deployment

Write your Streamlit app. Run it from a terminal after this cell executes.

In [ ]:
# ── Write agent.py ────────────────────────────────────────
DOMAIN_NAME        = "NovaTech HR Assistant — Aria"
DOMAIN_DESCRIPTION = "24/7 HR policy assistant for NovaTech Solutions employees"
KB_TOPICS          = [d["topic"] for d in DOCUMENTS]

agent_py = (
    "# agent.py - NovaTech Solutions HR Policy Bot\n"
    "# Standalone module. Import build_agent() in capstone_streamlit.py\n\n"
    "import os, datetime\n"
    "from dotenv import load_dotenv\n"
    "from typing import TypedDict, List\n"
    "import chromadb\n"
    "from sentence_transformers import SentenceTransformer\n"
    "from langchain_groq import ChatGroq\n"
    "from langchain_core.messages import SystemMessage, HumanMessage, AIMessage\n"
    "from langgraph.graph import StateGraph, END\n"
    "from langgraph.checkpoint.memory import MemorySaver\n\n"
    "load_dotenv()\n\n"
    "DOCUMENTS = [\n"
    + "".join(
        f'    {{"id": "{d["id"]}", "topic": "{d["topic"]}", "text": {repr(d["text"])}}},\n'
        for d in DOCUMENTS
    )
    + "]\n\n"
    "FAITHFULNESS_THRESHOLD = 0.7\n"
    "MAX_EVAL_RETRIES       = 2\n\n"
    "class CapstoneState(TypedDict):\n"
    "    question: str\n"
    "    messages: List[dict]\n"
    "    route: str\n"
    "    retrieved: str\n"
    "    sources: List[str]\n"
    "    tool_result: str\n"
    "    answer: str\n"
    "    faithfulness: float\n"
    "    eval_retries: int\n"
    "    employee_id: str\n"
    "    department: str\n\n"
)

# Append build_agent function as a raw string
agent_py += r'''
def build_agent():
    llm      = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    client   = chromadb.Client()
    try: client.delete_collection("novatech_hr_kb")
    except Exception: pass
    collection = client.create_collection("novatech_hr_kb")
    texts = [d["text"] for d in DOCUMENTS]
    collection.add(documents=texts, embeddings=embedder.encode(texts).tolist(),
                   ids=[d["id"] for d in DOCUMENTS],
                   metadatas=[{"topic": d["topic"]} for d in DOCUMENTS])

    def memory_node(state):
        msgs = state.get("messages", []) + [{"role": "user", "content": state["question"]}]
        return {"messages": msgs[-6:]}

    def router_node(state):
        q = state["question"]; msgs = state.get("messages", [])
        recent = "; ".join(f"{m['role']}: {m['content'][:60]}" for m in msgs[-3:-1]) or "none"
        prompt = (
            "You are a router for Aria, NovaTech HR Assistant.\n"
            "Options:\n"
            "- retrieve: HR policy questions (leave, payroll, benefits, appraisal, probation, exit, referral, wellness, conduct)\n"
            "- memory_only: follow-up about what was just said\n"
            "- tool: date-based calculations (leave balance, notice period end, upcoming deadlines)\n"
            f"Conversation: {recent}\nQuestion: {q}\nReply ONE word: retrieve / memory_only / tool"
        )
        d = llm.invoke(prompt).content.strip().lower()
        if "memory" in d: d = "memory_only"
        elif "tool" in d: d = "tool"
        else:             d = "retrieve"
        return {"route": d}

    def retrieval_node(state):
        res    = collection.query(query_embeddings=embedder.encode([state["question"]]).tolist(), n_results=3)
        topics = [m["topic"] for m in res["metadatas"][0]]
        ctx    = "\n\n---\n\n".join(f"[{topics[i]}]\n{res['documents'][0][i]}" for i in range(len(topics)))
        return {"retrieved": ctx, "sources": topics}

    def skip_retrieval_node(state):
        return {"retrieved": "", "sources": []}

    def tool_node(state):
        q = state["question"].lower(); today = datetime.date.today()
        try:
            if any(w in q for w in ["notice period", "last working day", "resign"]):
                end = today + datetime.timedelta(days=60)
                r = f"Resignation today ({today}) -> notice ends {end.strftime('%A, %B %d, %Y')} (60 days)."
            elif any(w in q for w in ["accrued", "leave balance", "leave days", "how many leave", "how much leave"]):
                acc = round(min(20.0, ((today.month-1) + today.day/30.0) * (20.0/12.0)), 1)
                r   = (f"As of {today}: Annual Leave accrued = {acc}/20 days. "
                       f"CL = 8/yr. SL = 8/yr. Mental Health Days = 4/yr. "
                       f"Check HR portal for personal balance after leaves taken.")
            elif any(w in q for w in ["deadline", "upcoming", "next hr"]):
                yr = today.year
                ms = [(datetime.date(yr,1,31),"Form 12BB deadline"),
                      (datetime.date(yr,3,31),"L&D budget lapses"),
                      (datetime.date(yr,4,1), "Insurance year renews"),
                      (datetime.date(yr,7,1), "Mid-year review opens"),
                      (datetime.date(yr,12,31),"CL/SL balances lapse")]
                coming = sorted([(d,l) for d,l in ms if d>=today])[:4]
                r = "Upcoming HR deadlines:\n" + "\n".join(f"  {d} ({(d-today).days}d) — {l}" for d,l in coming)
            else:
                r = f"Today is {today.strftime('%A, %B %d, %Y')} (Q{(today.month-1)//3+1} {today.year})."
        except Exception as e:
            r = f"Calculator error: {e}. Contact hr@novatech.in."
        return {"tool_result": r}

    def answer_node(state):
        q=state["question"]; ret=state.get("retrieved",""); tool=state.get("tool_result","")
        msgs=state.get("messages",[]); retries=state.get("eval_retries",0)
        parts = []
        if ret:  parts.append(f"POLICY KB:\n{ret}")
        if tool: parts.append(f"CALCULATOR:\n{tool}")
        ctx = "\n\n".join(parts)
        if ctx:
            sys_msg = (f"You are Aria, NovaTech HR Assistant. Answer using ONLY the context below. "
                       f"If not in context say: I do not have that information. Contact hr@novatech.in.\n\n{ctx}")
        else:
            sys_msg = "You are Aria, NovaTech HR Assistant. Answer from conversation history or ask to rephrase."
        if retries > 0:
            sys_msg += "\n\nPrevious answer failed quality check. Use ONLY context above."
        lc = [SystemMessage(content=sys_msg)]
        for m in msgs[:-1]:
            lc.append(HumanMessage(content=m["content"]) if m["role"]=="user" else AIMessage(content=m["content"]))
        lc.append(HumanMessage(content=q))
        return {"answer": llm.invoke(lc).content}

    def eval_node(state):
        ans=state.get("answer",""); ctx=state.get("retrieved","")[:500]; r=state.get("eval_retries",0)
        if not ctx: return {"faithfulness": 1.0, "eval_retries": r+1}
        try:
            score = float(llm.invoke(
                f"Rate faithfulness 0.0-1.0 (number only).\nContext:{ctx}\nAnswer:{ans[:300]}"
            ).content.strip().split()[0])
            score = max(0.0, min(1.0, score))
        except Exception:
            score = 0.5
        return {"faithfulness": score, "eval_retries": r+1}

    def save_node(state):
        return {"messages": state.get("messages",[]) + [{"role":"assistant","content":state["answer"]}]}

    def route_decision(state):
        r = state.get("route", "retrieve")
        return "tool" if r=="tool" else "skip" if r=="memory_only" else "retrieve"

    def eval_decision(state):
        return "save" if (state.get("faithfulness",1.0)>=FAITHFULNESS_THRESHOLD or
                          state.get("eval_retries",0)>=MAX_EVAL_RETRIES) else "answer"

    g = StateGraph(CapstoneState)
    for name, fn in [("memory",memory_node),("router",router_node),("retrieve",retrieval_node),
                     ("skip",skip_retrieval_node),("tool",tool_node),("answer",answer_node),
                     ("eval",eval_node),("save",save_node)]:
        g.add_node(name, fn)
    g.set_entry_point("memory")
    g.add_edge("memory","router")
    g.add_conditional_edges("router", route_decision, {"retrieve":"retrieve","skip":"skip","tool":"tool"})
    g.add_edge("retrieve","answer"); g.add_edge("skip","answer"); g.add_edge("tool","answer")
    g.add_edge("answer","eval")
    g.add_conditional_edges("eval", eval_decision, {"answer":"answer","save":"save"})
    g.add_edge("save", END)
    app = g.compile(checkpointer=MemorySaver())
    return app, embedder, collection
'''

with open("agent.py", "w", encoding="utf-8") as f:
    f.write(agent_py)
print("agent.py written")

# ── Write capstone_streamlit.py ────────────────────────────
streamlit_py = r'''# capstone_streamlit.py - NovaTech HR Assistant (Aria)
# Run: streamlit run capstone_streamlit.py
import streamlit as st
import uuid
from agent import build_agent, DOCUMENTS

DOMAIN_NAME  = "NovaTech HR Assistant"
KB_TOPICS    = [d["topic"] for d in DOCUMENTS]

st.set_page_config(page_title=DOMAIN_NAME, page_icon="🏢", layout="centered")
st.title("🏢 " + DOMAIN_NAME)
st.caption("Hi, I am Aria — your 24/7 HR policy companion at NovaTech Solutions.")

@st.cache_resource
def load_agent():
    return build_agent()

try:
    agent_app, embedder, collection = load_agent()
    st.success(f"Knowledge base loaded — {collection.count()} HR policy documents")
except Exception as e:
    st.error(f"Failed to load agent: {e}")
    st.stop()

if "messages" not in st.session_state:
    st.session_state.messages = []
if "thread_id" not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())[:8]

with st.sidebar:
    st.header("About Aria")
    st.write(
        "Aria answers questions about NovaTech HR policies, benefits, leave rules, "
        "payroll, appraisals, and more — grounded strictly in the company handbook."
    )
    st.write(f"Session: {st.session_state.thread_id}")
    st.divider()
    st.write("**HR Topics Covered:**")
    for t in KB_TOPICS:
        st.write(f"• {t}")
    st.divider()
    if st.button("New Conversation"):
        st.session_state.messages = []
        st.session_state.thread_id = str(uuid.uuid4())[:8]
        st.rerun()

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

if prompt := st.chat_input("Ask me anything about NovaTech HR policies..."):
    with st.chat_message("user"):
        st.write(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt})

    with st.chat_message("assistant"):
        with st.spinner("Checking HR policies..."):
            config = {"configurable": {"thread_id": st.session_state.thread_id}}
            result = agent_app.invoke({"question": prompt}, config=config)
            answer = result.get("answer", "Sorry, I could not answer. Please contact hr@novatech.in.")
        st.write(answer)
        faith   = result.get("faithfulness", 0.0)
        sources = result.get("sources", [])
        if sources:
            st.caption(f"Sources: {', '.join(sources)} | Faithfulness: {faith:.2f}")

    st.session_state.messages.append({"role": "assistant", "content": answer})
'''

with open("capstone_streamlit.py", "w", encoding="utf-8") as f:
    f.write(streamlit_py)
print("capstone_streamlit.py written")
print()
print("Run: streamlit run capstone_streamlit.py")

agent.py written
capstone_streamlit.py written

Run: streamlit run capstone_streamlit.py


---
## Part 8 — Written Summary (Required)

Fill in the markdown cell below. This is submitted along with your notebook.

## My Capstone Summary

**Name:** Kalisayar Mukherjee

**Roll:** 23051915

**Email:** 23051915@kiit.ac.in

**Domain chosen:** HR Policy Bot — NovaTech Solutions Pvt. Ltd.

**What the agent does:** Aria is a 24/7 HR Policy Assistant for employees of NovaTech Solutions. It answers questions about leave policies, payroll, performance reviews, health insurance, probation, resignation, referral bonuses, mental wellness, and the code of conduct — all strictly grounded in the company handbook. For time-sensitive queries (leave balance, notice period end date, upcoming HR deadlines) it uses a custom Leave Balance Calculator tool that performs live date arithmetic.

**Knowledge base:** 12 documents covering: Annual Leave Policy, Casual and Sick Leave, Remote Work and Flexible Hours, Payroll and Compensation, Performance Review and Appraisal, Learning and Development Budget, Health Insurance and Benefits, Probation and Confirmation, Resignation and Exit Process, Employee Referral Bonus, Mental Wellness and Mental Health Days, Code of Conduct and Anti-Harassment Policy.

**Tool used:** Leave Balance Calculator — a custom datetime tool that computes accrued annual leave based on today's date, calculates when a 60-day notice period ends, and lists upcoming HR deadlines (L&D budget lapse, insurance renewal, appraisal cycles, TDS submission). Domain-critical because leave balance queries depend on the current date — a static knowledge base cannot answer them correctly.

**RAGAS baseline scores:**
- Faithfulness: 1.000
- Answer Relevance: 0.807
- Context Precision: 1.000

**Test results:** 10 / 10 tests passed. Red-team: 2 / 2 passed.

**One thing I would improve with more time:** I would replace the hand-written HR documents with a PDF ingestion pipeline using PyMuPDF to load actual company policy PDFs page by page. This would allow the knowledge base to stay synchronized with real policy updates automatically and would improve retrieval precision by preserving numbered clauses, table structures, and section headers from original documents — details lost in free-form prose.

**Most surprising thing I learned building this:** The router node has a disproportionate impact on overall faithfulness. When the router incorrectly sends a calculation query to retrieval (or vice versa), the answer node produces unfaithful responses even when the LLM and KB are both high quality. Writing clear, example-driven routing instructions proved as important as the knowledge base content itself.

---
## Submission Checklist

Before submitting, verify each item:

- ✅ All TODO sections in the notebook have been filled in
- ✅ Knowledge base has at least 10 documents
- ✅ All cells run without errors (Kernel → Restart & Run All)
- ✅ Test suite shows results for all 10 questions
- ✅ RAGAS baseline scores are recorded
- ✅ `capstone_streamlit.py` runs and the chat UI works
- ✅ Conversation memory works — ask 3 follow-up questions in one session
- ✅ Written summary is complete

**Deliverables:**
1. This completed notebook (`day13_capstone.ipynb`)
2. `capstone_streamlit.py` (or `capstone_api.py` for FastAPI)
3. `agent.py` (your shared agent module)

---
*You have built 12 days of skills. This is where they come together. Go make something real.*